# CVAE Analysis notebook

This notebook orchestrates analysis using the refactored package `vae_timbre_spaces`.
- Loads a trained checkpoint
- Extracts embeddings (μ, logvar)
- Runs PCA + UMAP visualizations
- Computes silhouette scores
- Performs latent interpolation and saves audio

All outputs are saved under `outputs/`.

In [1]:
# Imports
import sys
from pathlib import Path
import glob
import json
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# package imports (do NOT redefine models/dataset)
from vae_timbre_spaces.models import ConditionalVAE
from vae_timbre_spaces.dataset import NsynthMelCacheDataset, load_examples_json
from vae_timbre_spaces.analysis.embeddings import extract_mu_logvar, save_embeddings
from vae_timbre_spaces.analysis.umap import fit_umap, transform_umap
from vae_timbre_spaces.analysis.silhouette import silhouette_by_family, run_silhouette_for_pitch_window
from vae_timbre_spaces.analysis.interpolation import interpolate_between_mus, decode_from_z

print('Imports OK')

/home/satan/miniconda3/envs/timbre_space_dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [29]:
from datetime import datetime
from pathlib import Path

# Cria um id único por execução
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Ex: outputs/runs/20260218_153012/audio
RUNS_DIR = Path("outputs") / "runs"
AUDIO_RUN_DIR = RUNS_DIR / RUN_ID / "audio"
AUDIO_RUN_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_RUN_DIR = RUNS_DIR / RUN_ID / "plots"
EMB_RUN_DIR   = RUNS_DIR / RUN_ID / "embeddings"
PLOTS_RUN_DIR.mkdir(parents=True, exist_ok=True)
EMB_RUN_DIR.mkdir(parents=True, exist_ok=True)


print("Run folder:", AUDIO_RUN_DIR)
print("Run folder:", PLOTS_RUN_DIR)
print("Run folder:", EMB_RUN_DIR)


Run folder: outputs/runs/20260218_190030/audio
Run folder: outputs/runs/20260218_190030/plots
Run folder: outputs/runs/20260218_190030/embeddings


In [ ]:
# # Paths & device
# PROJECT_ROOT = Path().resolve().parent
CKPT_DIR = PROJECT_ROOT / 'notebooks' / 'ckpts'
# OUT_DIR = PROJECT_ROOT / 'outputs'
# PLOTS_DIR = OUT_DIR / 'plots'
# AUDIO_DIR = OUT_DIR / 'audio'

# for p in (OUT_DIR, PLOTS_DIR, AUDIO_DIR):
#     p.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [3]:
# 1) Locate checkpoint (pick latest if multiple)
ckpt_paths = sorted(CKPT_DIR.glob('*.pt'), key=lambda p: p.stat().st_mtime)
if len(ckpt_paths) == 0:
    raise FileNotFoundError(f'No checkpoints found in {CKPT_DIR}')

CKPT_PATH = ckpt_paths[-1]
print('Using checkpoint:', CKPT_PATH)

ckpt = torch.load(CKPT_PATH, map_location=device)
config = ckpt.get('config', {})
print('Checkpoint config keys:', list(config.keys()))

Using checkpoint: /home/satan/git/VAE-Timbre-Spaces/notebooks/ckpts/cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15.pt


/tmp/ipykernel_65782/3172288365.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location=device)


Checkpoint config keys: ['LATENT_DIM', 'BETA_MAX', 'FREE_BITS', 'WARMUP_STEPS', 'LR', 'cond_dim', 'pitch_vocab', 'N_MELS', 'N_FFT', 'HOP', 'T', 'SR']


In [4]:
print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else "Not a dict")

print(type(ckpt['model_state']))
print(list(ckpt['model_state'].keys())[:5])

print(CKPT_PATH)

<class 'dict'>
dict_keys(['model_state', 'optimizer_state', 'global_step', 'epoch', 'config'])
<class 'collections.OrderedDict'>
['encoder.conv1.weight', 'encoder.conv1.bias', 'encoder.conv2.weight', 'encoder.conv2.bias', 'encoder.conv3.weight']
/home/satan/git/VAE-Timbre-Spaces/notebooks/ckpts/cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15.pt


In [5]:
ckpt = torch.load(CKPT_PATH, map_location=device)

print("Keys:", ckpt.keys())
print("model_state length:", len(ckpt["model_state"]))


Keys: dict_keys(['model_state', 'optimizer_state', 'global_step', 'epoch', 'config'])
model_state length: 19


/tmp/ipykernel_65782/3558755710.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location=device)


In [6]:
# 2) Instantiate model from checkpoint config
ckpt = torch.load(CKPT_PATH, map_location=device)

# Detect format automatically
if isinstance(ckpt, dict) and "model_state" in ckpt:
    state_dict = ckpt["model_state"]
    config = ckpt.get("config", {})
else:
    state_dict = ckpt
    config = {}

cvae = ConditionalVAE(
    latent_dim=int(config.get("LATENT_DIM", 32)),
    pitch_vocab=int(config.get("pitch_vocab", 128)),
    cond_dim=int(config.get("cond_dim", 16)),
).to(device)

cvae.load_state_dict(state_dict)
cvae.eval()

print("Model loaded from:", CKPT_PATH)


Model loaded from: /home/satan/git/VAE-Timbre-Spaces/notebooks/ckpts/cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15.pt


/tmp/ipykernel_65782/1946768830.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location=device)


In [7]:
# 3) Prepare test dataloader
SPLIT = 'test'  # change to 'valid' if desired
split_root = PROJECT_ROOT / 'data' / f'nsynth-{SPLIT}.jsonwav' / f'nsynth-{SPLIT}'
# fallback if original layout different
if not split_root.exists():
    # try the notebook's TRAIN/VALID/TEST_ROOT variables path
    split_root = PROJECT_ROOT / 'data' / f'nsynth-{SPLIT}'

print('split_root:', split_root)
keys, examples = load_examples_json(split_root)
print('num examples:', len(keys))

cache_dir = PROJECT_ROOT / 'data' / 'nsynth_mel_cache' / SPLIT
print('cache_dir:', cache_dir)

dataset = NsynthMelCacheDataset(keys, examples, cache_dir)
loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=(device.type=='cuda'))

split_root: /home/satan/git/VAE-Timbre-Spaces/data/nsynth-test.jsonwav/nsynth-test
num examples: 4096
cache_dir: /home/satan/git/VAE-Timbre-Spaces/data/nsynth_mel_cache/test


In [32]:
# 4) Extract embeddings (mu, logvar, pitch, family, keys)
N_SAMPLES = min(20000, len(keys))
mu_all, logvar_all, pitch_all, family_all, keys_all = extract_mu_logvar(cvae, loader, n_samples=N_SAMPLES, device=device)
print('mu_all shape:', mu_all.shape)

# Save embeddings
model_name = CKPT_PATH.stem
print(EMB_RUN_DIR)
emb_path = save_embeddings(mu_all, logvar_all, pitch_all, family_all, keys_all, split=SPLIT, model_name=model_name, out_dir=EMB_RUN_DIR)
print('Saved embeddings to:', emb_path)

mu_all shape: (4096, 64)
outputs/runs/20260218_190030/embeddings
Saved embeddings to: outputs/runs/20260218_190030/embeddings/embeddings_test_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15.npz


In [18]:
FAMILY_ID_TO_NAME = {
    0: "bass",
    1: "brass",
    2: "flute",
    3: "guitar",
    4: "keyboard",
    5: "mallet",
    6: "organ",
    7: "reed",
    8: "string",
    9: "synth_lead",
    10: "vocal",
}

def family_to_name(fid: int) -> str:
    return FAMILY_ID_TO_NAME.get(int(fid), f"family_{int(fid)}")


In [30]:
pca = PCA(n_components=2, random_state=42)
mu_pca = pca.fit_transform(mu_all)

df_pca = pd.DataFrame({
    "pc1": mu_pca[:, 0],
    "pc2": mu_pca[:, 1],
    "pitch": pitch_all.astype(int),
    "family": family_all.astype(int),
    "family_name": [family_to_name(f) for f in family_all],
    "key": keys_all,
})

fig = px.scatter(
    df_pca,
    x="pc1", y="pc2",
    color="family_name",
    hover_data=["family", "pitch", "key"],
    title=f"PCA (mu) — {model_name} — {SPLIT}",
)

html_path = PLOTS_RUN_DIR / f"pca_{model_name}_{SPLIT}.html"
fig.write_html(str(html_path))
fig.show()
print("Saved PCA plot:", html_path)

try:
    fig.write_image(str(PLOTS_RUN_DIR / f"pca_{model_name}_{SPLIT}.png"))
except Exception:
    print("Could not write PNG (kaleido may be missing). HTML saved.")


Saved PCA plot: outputs/runs/20260218_190030/plots/pca_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test.html
Could not write PNG (kaleido may be missing). HTML saved.


In [28]:
# 6) UMAP visualization
reducer, mu_umap = fit_umap(mu_all, n_neighbors=30, min_dist=0.1, random_state=42)

df_umap = pd.DataFrame({
    "u1": mu_umap[:, 0],
    "u2": mu_umap[:, 1],
    "pitch": pitch_all.astype(int),
    "family": family_all.astype(int),
    "family_name": [family_to_name(f) for f in family_all],
    "key": keys_all,
})

fig = px.scatter(
    df_umap,
    x="u1", y="u2",
    color="family_name",
    hover_data=["family", "pitch", "key"],
    title=f"UMAP (mu) — {model_name} — {SPLIT}",
)

html_path = PLOTS_RUN_DIR / f"umap_{model_name}_{SPLIT}.html"
fig.write_html(str(html_path))
fig.show()
print("Saved UMAP plot:", html_path)

try:
    fig.write_image(str(PLOTS_RUN_DIR / f"umap_{model_name}_{SPLIT}.png"))
except Exception:
    print("Could not write PNG (kaleido may be missing). HTML saved.")


/home/satan/miniconda3/envs/timbre_space_dl/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Saved UMAP plot: outputs/runs/20260218_185903/plots/umap_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test.html
Could not write PNG (kaleido may be missing). HTML saved.


In [ ]:
# 7) Silhouette: global per family
sil_global_df = silhouette_by_family(mu_all, family_all)
csv_path = PLOTS_RUN_DIR / f'silhouette_global_{model_name}_{SPLIT}.csv'
sil_global_df.to_csv(csv_path)
print('Saved silhouette table:', csv_path)

# quick bar plot
fig = px.bar(sil_global_df.reset_index(), x='family', y='mean_silhouette', error_y='std_silhouette', title=f'Silhouette by family — {model_name} — {SPLIT}')
fig.show()
fig.write_html(str(PLOTS_RUN_DIR / f'silhouette_global_{model_name}_{SPLIT}.html'))

Saved silhouette table: /home/satan/git/VAE-Timbre-Spaces/outputs/plots/silhouette_global_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test.csv


In [ ]:
# Pitch-window silhouette example: pitch 60 ± 1
pitch_center = 60
sil_pitch_df = run_silhouette_for_pitch_window(mu_all, pitch_all, family_all, pitch_center=pitch_center, tolerance=1, min_samples_per_family=10)
path_csv = PLOTS_RUN_DIR / f'silhouette_pitch{pitch_center}_{model_name}_{SPLIT}.csv'
sil_pitch_df.to_csv(path_csv)
print('Saved pitch-window silhouette:', path_csv)

fig = px.bar(sil_pitch_df.reset_index(), x='family', y='mean_silhouette', error_y='std_silhouette', title=f'Silhouette by family (pitch {pitch_center}±1) — {model_name} — {SPLIT}')
fig.show()
fig.write_html(str(PLOTS_RUN_DIR / f'silhouette_pitch{pitch_center}_{model_name}_{SPLIT}.html'))

Saved pitch-window silhouette: /home/satan/git/VAE-Timbre-Spaces/outputs/plots/silhouette_pitch60_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test.csv


In [33]:
# 8) Interpolation: pick two samples by family name (string) and generate audio
import numpy as np
import torch
import soundfile as sf
import librosa

# -------------------------
# 0) Family mapping (NSynth)
# -------------------------
FAMILY_ID_TO_NAME = {
    0: "bass",
    1: "brass",
    2: "flute",
    3: "guitar",
    4: "keyboard",
    5: "mallet",
    6: "organ",
    7: "reed",
    8: "string",
    9: "synth_lead",
    10: "vocal",
}
FAMILY_NAME_TO_ID = {v: k for k, v in FAMILY_ID_TO_NAME.items()}

def family_to_id(name_or_id):
    if isinstance(name_or_id, str):
        name = name_or_id.strip().lower()
        if name not in FAMILY_NAME_TO_ID:
            raise ValueError(f"Unknown family '{name_or_id}'. Options: {sorted(FAMILY_NAME_TO_ID.keys())}")
        return FAMILY_NAME_TO_ID[name]
    return int(name_or_id)

# -------------------------
# 1) Choose interpolation endpoints (EDIT HERE)
# -------------------------
SRC_FAMILY = "mallet"
DST_FAMILY = "vocal"

# optional: keep pitch consistent (recommended)
USE_PITCH_FILTER = True
PITCH_TARGET = 60
PITCH_TOL = 1

# optional: refine by key substring (useful when you want a specific instrument)
# example: "keyboard_acoustic" or "vocal_synthetic"
SRC_KEY_CONTAINS = None
DST_KEY_CONTAINS = None

# how to pick among candidates
PICK_MODE = "random"   # "random" | "first"
SEED = 42

# interpolation steps
n_steps = 9

# -------------------------
# 2) Find indices idx_a, idx_b
# -------------------------
src_id = family_to_id(SRC_FAMILY)
dst_id = family_to_id(DST_FAMILY)

idx_src = np.where(family_all.astype(int) == src_id)[0]
idx_dst = np.where(family_all.astype(int) == dst_id)[0]

if USE_PITCH_FILTER:
    p = pitch_all.astype(int)
    idx_src = idx_src[np.abs(p[idx_src] - PITCH_TARGET) <= PITCH_TOL]
    idx_dst = idx_dst[np.abs(p[idx_dst] - PITCH_TARGET) <= PITCH_TOL]

if SRC_KEY_CONTAINS:
    idx_src = np.array([i for i in idx_src if SRC_KEY_CONTAINS in str(keys_all[i])], dtype=int)
if DST_KEY_CONTAINS:
    idx_dst = np.array([i for i in idx_dst if DST_KEY_CONTAINS in str(keys_all[i])], dtype=int)

if len(idx_src) == 0:
    raise RuntimeError(f"No samples for SRC_FAMILY={SRC_FAMILY} after filters. Try disabling pitch/key filters.")
if len(idx_dst) == 0:
    raise RuntimeError(f"No samples for DST_FAMILY={DST_FAMILY} after filters. Try disabling pitch/key filters.")

rng = np.random.default_rng(SEED)
if PICK_MODE == "random":
    idx_a = int(rng.choice(idx_src))
    idx_b = int(rng.choice(idx_dst))
else:
    idx_a = int(idx_src[0])
    idx_b = int(idx_dst[0])

print("[selected A]",
      "family:", SRC_FAMILY, f"(id={src_id})",
      "| pitch:", int(pitch_all[idx_a]),
      "| key:", keys_all[idx_a])
print("[selected B]",
      "family:", DST_FAMILY, f"(id={dst_id})",
      "| pitch:", int(pitch_all[idx_b]),
      "| key:", keys_all[idx_b])

# Keep pitch fixed (you can choose A or B; usually keep A)
pitch_interp = int(pitch_all[idx_a])

# -------------------------
# 3) Interpolate in latent (mu) and decode
# -------------------------
zA = mu_all[idx_a].astype(np.float32)
zB = mu_all[idx_b].astype(np.float32)

mus, alphas = interpolate_between_mus(zA, zB, n_steps=n_steps)

Z_t = torch.tensor(np.stack(mus, axis=0), device=device, dtype=torch.float32)
pitch_t = torch.full((n_steps,), pitch_interp, device=device, dtype=torch.long)

with torch.no_grad():
    x_hat = decode_from_z(cvae, Z_t, pitch_t)  # (n_steps, 1, 80, 128)

# -------------------------
# 4) Helper: norm -> db -> audio (Griffin-Lim)
# -------------------------
def norm_to_logmel_db(x_norm: np.ndarray) -> np.ndarray:
    x01 = (x_norm + 1.0) / 2.0
    return x01 * 80.0 - 80.0

SR = 16000
N_FFT = 1024
HOP = 256
WIN = 1024

for i in range(n_steps):
    xhat_norm = x_hat[i, 0].detach().cpu().numpy()
    xhat_db = norm_to_logmel_db(xhat_norm)

    mel_power = librosa.db_to_power(xhat_db, ref=1.0)
    try:
        stft_mag = librosa.feature.inverse.mel_to_stft(
            M=mel_power, sr=SR, n_fft=N_FFT, power=1.0
        )
        wav = librosa.griffinlim(
            stft_mag, n_iter=64, hop_length=HOP, win_length=WIN
        )
    except Exception:
        wav = librosa.feature.inverse.mel_to_audio(
            M=mel_power, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=WIN, n_iter=32, power=1.0
        )

    out_path = AUDIO_RUN_DIR / f"interp_{model_name}_{SPLIT}_{SRC_FAMILY}_to_{DST_FAMILY}_p{pitch_interp}_a{alphas[i]:.2f}_{i}.wav"
    sf.write(str(out_path), wav, SR)
    print("Saved:", out_path)


[selected A] family: mallet (id=5) | pitch: 59 | key: mallet_acoustic_062-059-127
[selected B] family: vocal (id=10) | pitch: 59 | key: vocal_acoustic_000-059-100
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p59_a0.00_0.wav
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p59_a0.12_1.wav
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p59_a0.25_2.wav
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p59_a0.38_3.wav
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p59_a0.50_4.wav
Saved: outputs/runs/20260218_190030/audio/interp_cvae_pitch_lat64_beta1.0_fb0.25_20260218_181117_epoch15_test_mallet_to_vocal_p